Đọc dữ liệu

In [1]:
# Nạp thư viện os (Operating System) — dùng để tương tác với hệ điều hành (ví dụ: quản lý tệp tin, nối đường dẫn thư mục, kiểm tra tệp tồn tại).
import os
# Nạp thư viện numpy và đặt tên viết tắt là np — dùng để xử lý tính toán đại số tuyến tính, mảng nhiều chiều (array) và các phép toán số học tốc độ cao.
import numpy as np
# Nạp thư viện pandas và đặt tên viết tắt là pd — dùng để làm việc với dữ liệu dạng bảng (DataFrame), hỗ trợ đọc/ghi tệp CSV, Excel và tiền xử lý dữ liệu
import pandas as pd

# Ghép đường dẫn giữa thư mục 'path' (từ kagglehub) và tên tệp CSV
file_path = os.path.join(
    "..", "..", "data", "raw", "Diamonds Prices2022.csv"
)

# Đọc tệp CSV vào biến 'diamonds' (DataFrame) với dấu phân cách các cột là dấu phẩy (,)
diamonds = pd.read_csv(file_path, sep=',', index_col=0)

# In kích thước của tập dữ liệu (số dòng, số cột)
print(f"Kích thước tập dữ liệu ban đầu: {diamonds.shape}\n")

# Hiển thị thông tin tổng quan
diamonds.info()

# In tiêu đề thông báo cho phần hiển thị 5 dòng đầu
print("\n--- 5 dòng đầu tiên ---")
# Hiển thị 5 dòng dữ liệu đầu tiên
print(diamonds.head())

# In tiêu đề thông báo cho phần kiểm tra các giá trị phân loại
print("\n--- Giá trị phân loại của các thuộc tính dạng chuỗi ---")
# Lấy và in danh sách các giá trị duy nhất (không trùng lặp) của cột 'cut' (vết cắt)
print(f"Cut: {diamonds['cut'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'color' (màu sắc)
print(f"Color: {diamonds['color'].unique()}")
# Lấy và in danh sách các giá trị duy nhất của cột 'clarity' (độ tinh khiết)
print(f"Clarity: {diamonds['clarity'].unique()}")

# In tiêu đề thông báo cho phần thống kê mô tả
print("\n--- Thống kê mô tả ---")
# Tính toán các chỉ số thống kê cơ bản cho các cột số
print(diamonds.describe())

Kích thước tập dữ liệu ban đầu: (53943, 10)

<class 'pandas.DataFrame'>
RangeIndex: 53943 entries, 1 to 53943
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53943 non-null  float64
 1   cut      53943 non-null  str    
 2   color    53943 non-null  str    
 3   clarity  53943 non-null  str    
 4   depth    53943 non-null  float64
 5   table    53943 non-null  float64
 6   price    53943 non-null  int64  
 7   x        53943 non-null  float64
 8   y        53943 non-null  float64
 9   z        53943 non-null  float64
dtypes: float64(6), int64(1), str(3)
memory usage: 4.1 MB

--- 5 dòng đầu tiên ---
   carat      cut color clarity  depth  table  price     x     y     z
1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
4   0.29  Premium     I     VS2   

Tiền xử lý

In [2]:
# Nạp công cụ OrdinalEncoder từ thư viện scikit-learn để mã hóa các biến phân loại có thứ tự
from sklearn.preprocessing import OrdinalEncoder

# 1: Loại bỏ thuộc tính không mang thông tin
# Kiểm tra xem cột số thứ tự thừa 'Unnamed: 0' có tồn tại trong DataFrame hay không
if 'Unnamed: 0' in diamonds.columns:
    # Nếu có, xóa bỏ cột này khỏi tập dữ liệu
    diamonds = diamonds.drop(columns=['Unnamed: 0'])

# 2: Loại bỏ dữ liệu nhiễu (kích thước x, y, z <= 0)
# Lấy tổng số dòng dữ liệu ban đầu trước khi lọc
initial_rows = len(diamonds)

# Lọc giữ lại các dòng có cả 3 kích thước (x, y, z) đều lớn hơn 0 và tạo bản sao dữ liệu mới
diamonds = diamonds[(diamonds['x'] > 0) & (diamonds['y'] > 0) & (diamonds['z'] > 0)].copy()

# Tính số lượng dòng dữ liệu bị loại bỏ (bằng số dòng ban đầu trừ số dòng sau khi lọc)
invalid_dimensions = initial_rows - len(diamonds)

# In số lượng dòng nhiễu đã bị loại bỏ ra màn hình (thêm \n ở đầu để cách dòng trên)
print(f"\nĐã loại bỏ {invalid_dimensions} dòng dữ liệu nhiễu (x, y, z <= 0).")

# 3: Loại bỏ các dòng trùng lặp hoàn toàn
# Đếm tổng số dòng dữ liệu bị lặp lại hoàn toàn trong tập dữ liệu
num_duplicates = diamonds.duplicated().sum()

# Xóa bỏ các dòng trùng lặp, chỉ giữ lại 1 bản ghi đại diện
diamonds = diamonds.drop_duplicates()

# In số lượng dòng trùng lặp đã xóa ra màn hình (thêm \n ở đầu để cách với dòng trên)
print(f"\nĐã loại bỏ {num_duplicates} dòng trùng lặp.")

# 4: Mã hóa thứ tự (Ordinal Encoding) cho các thuộc tính phân loại
# Định nghĩa danh sách các mức độ chất lượng của vết cắt 'cut' theo thứ tự tăng dần (Fair là thấp nhất, Ideal là cao nhất)
cut_categories = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

# Định nghĩa danh sách màu sắc 'color' theo thứ tự tăng dần từ kém nhất đến tốt nhất (J -> D)
color_categories = ['J', 'I', 'H', 'G', 'F', 'E', 'D']

# Định nghĩa danh sách độ tinh khiết 'clarity' theo thứ tự tăng dần từ kém nhất đến tốt nhất (I1 -> IF)
clarity_categories = ['I1', 'SI2', 'SI1', 'VS2', 'VS1', 'VVS2', 'VVS1', 'IF']

# Khởi tạo đối tượng OrdinalEncoder với danh sách thứ tự đã thiết lập cho từng cột
encoder = OrdinalEncoder(categories=[cut_categories, color_categories, clarity_categories])

# chuyển đổi các cột dữ liệu dạng chữ (phân loại) thành dạng số dựa trên thứ tự đã định nghĩa
diamonds[['cut', 'color', 'clarity']] = encoder.fit_transform(diamonds[['cut', 'color', 'clarity']])

# 5: Kiểm tra dữ liệu sau tiền xử lý
# In kích thước mới của tập dữ liệu (số dòng, số cột) sau khi đã làm sạch và chuyển đổi
print(f"\nKích thước tập dữ liệu sau tiền xử lý: {diamonds.shape}\n")

# In tiêu đề thông báo hiển thị 5 dòng đầu
print("--- 5 dòng dữ liệu đầu sau tiền xử lý ---")

# Hiển thị 5 dòng dữ liệu đầu tiên sau tiền xử lý
print(diamonds.head())


Đã loại bỏ 20 dòng dữ liệu nhiễu (x, y, z <= 0).

Đã loại bỏ 148 dòng trùng lặp.

Kích thước tập dữ liệu sau tiền xử lý: (53775, 10)

--- 5 dòng dữ liệu đầu sau tiền xử lý ---
   carat  cut  color  clarity  depth  table  price     x     y     z
1   0.23  4.0    5.0      1.0   61.5   55.0    326  3.95  3.98  2.43
2   0.21  3.0    5.0      2.0   59.8   61.0    326  3.89  3.84  2.31
3   0.23  1.0    5.0      4.0   56.9   65.0    327  4.05  4.07  2.31
4   0.29  3.0    1.0      3.0   62.4   58.0    334  4.20  4.23  2.63
5   0.31  1.0    0.0      1.0   63.3   58.0    335  4.34  4.35  2.75


Chia dữ liệu

In [3]:
from sklearn.model_selection import KFold

# X là dữ liệu các thuộc tính
X = diamonds.drop(columns=["price"])
# y là dữ liệu nhãn
y = diamonds["price"]

# Chia dữ liệu thành 5 fold
Kf = KFold(n_splits = 5, shuffle = True, random_state=42)

Training

In [4]:

# Import các thư viện cần thiết

# Import mô hình Support Vector Regression
from sklearn.svm import SVR

# Import công cụ chia dữ liệu theo K-Fold
from sklearn.model_selection import KFold

# Import công cụ chuẩn hóa dữ liệu
from sklearn.preprocessing import StandardScaler

# Import công cụ kết hợp tiền xử lý và mô hình
from sklearn.pipeline import make_pipeline

# Import các chỉ số đánh giá mô hình
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Thư viện tính toán số học
import numpy as np

# Thư viện xử lý dữ liệu
import pandas as pd

# Thư viện đo thời gian
import time

# Khởi tạo danh sách lưu kết quả đánh giá từng Fold
results = []

# Khởi tạo biến lưu tổng thời gian huấn luyện
total_training_time = 0


# Bắt đầu thực hiện 5-Fold Cross Validation
for fold, (train_index, test_index) in enumerate(Kf.split(X), 1):

    # Lấy dữ liệu huấn luyện theo chỉ số của Fold hiện tại
    X_train = X.iloc[train_index]
    y_train = y.iloc[train_index]

    # Lấy dữ liệu kiểm tra theo chỉ số của Fold hiện tại
    X_test = X.iloc[test_index]
    y_test = y.iloc[test_index]


    # Khởi tạo Pipeline gồm chuẩn hóa dữ liệu và mô hình SVR
    svr_model = make_pipeline(
        StandardScaler(),  # Chuẩn hóa các thuộc tính đầu vào
        SVR()              # Sử dụng SVR với toàn bộ tham số mặc định
    )


    # Bắt đầu tính thời gian huấn luyện
    start_time = time.time()

    # Huấn luyện mô hình trên tập Train
    # StandardScaler chỉ học thông số chuẩn hóa từ tập Train
    svr_model.fit(X_train, y_train)

    # Kết thúc tính thời gian huấn luyện
    training_time = time.time() - start_time

    # Cộng thời gian huấn luyện của Fold hiện tại vào tổng thời gian
    total_training_time += training_time


    # Dự đoán giá kim cương trên tập Train
    y_pred_train = svr_model.predict(X_train)

    # Dự đoán giá kim cương trên tập Test
    y_pred_test = svr_model.predict(X_test)


    # Tính MAE trên tập Train
    mae_train = mean_absolute_error(y_train, y_pred_train)

    # Tính MSE trên tập Train
    mse_train = mean_squared_error(y_train, y_pred_train)

    # Tính RMSE trên tập Train
    rmse_train = np.sqrt(mse_train)

    # Tính R2 trên tập Train
    r2_train = r2_score(y_train, y_pred_train)


    # Tính MAE trên tập Test
    mae_test = mean_absolute_error(y_test, y_pred_test)

    # Tính MSE trên tập Test
    mse_test = mean_squared_error(y_test, y_pred_test)

    # Tính RMSE trên tập Test
    rmse_test = np.sqrt(mse_test)

    # Tính R2 trên tập Test
    r2_test = r2_score(y_test, y_pred_test)


    # Lưu kết quả đánh giá của Fold hiện tại
    results.append([
        fold,
        mae_train, mse_train, rmse_train, r2_train,
        mae_test, mse_test, rmse_test, r2_test,
        training_time
    ])


    # In kết quả đánh giá của Fold hiện tại
    print(f"\n========== FOLD {fold} ==========")

    # In số lượng mẫu Train và Test
    print(f"Số mẫu Train: {len(X_train)}")
    print(f"Số mẫu Test: {len(X_test)}")

    # In kết quả Train
    print("\n--- Kết quả Train ---")
    print(f"MAE Train:  {mae_train:.4f}")
    print(f"MSE Train:  {mse_train:.4f}")
    print(f"RMSE Train: {rmse_train:.4f}")
    print(f"R2 Train:   {r2_train:.4f}")

    # In kết quả Test
    print("\n--- Kết quả Test ---")
    print(f"MAE Test:   {mae_test:.4f}")
    print(f"MSE Test:   {mse_test:.4f}")
    print(f"RMSE Test:  {rmse_test:.4f}")
    print(f"R2 Test:    {r2_test:.4f}")

    # In thời gian huấn luyện của Fold hiện tại
    print(f"\nThời gian Train: {training_time:.4f} giây")


# Chuyển danh sách kết quả thành DataFrame
columns = [
    "Fold",
    "MAE Train", "MSE Train", "RMSE Train", "R2 Train",
    "MAE Test", "MSE Test", "RMSE Test", "R2 Test",
    "Training Time"
]

results_df = pd.DataFrame(results, columns=columns)


# Tính giá trị trung bình của các chỉ số qua 5 Fold
mean_results = results_df.drop(columns=["Fold"]).mean()


# In kết quả trung bình của 5 Fold
print("\n========== KẾT QUẢ TRUNG BÌNH 5 FOLD ==========")

# In kết quả trung bình trên Train
print("\n--- Trung bình Train ---")
print(f"MAE Train:  {mean_results['MAE Train']:.4f}")
print(f"MSE Train:  {mean_results['MSE Train']:.4f}")
print(f"RMSE Train: {mean_results['RMSE Train']:.4f}")
print(f"R2 Train:   {mean_results['R2 Train']:.4f}")


# In kết quả trung bình trên Test
print("\n--- Trung bình Test ---")
print(f"MAE Test:   {mean_results['MAE Test']:.4f}")
print(f"MSE Test:   {mean_results['MSE Test']:.4f}")
print(f"RMSE Test:  {mean_results['RMSE Test']:.4f}")
print(f"R2 Test:    {mean_results['R2 Test']:.4f}")


# In thời gian huấn luyện trung bình của 5 Fold
print(f"\nThời gian Train trung bình: {mean_results['Training Time']:.4f} giây")

# In tổng thời gian huấn luyện của 5 Fold
print(f"Tổng thời gian Train: {total_training_time:.4f} giây")



========== FOLD 1 ==========
Số mẫu Train: 43020
Số mẫu Test: 10755

--- Kết quả Train ---
MAE Train:  1277.6673
MSE Train:  7396445.7606
RMSE Train: 2719.6407
R2 Train:   0.5351

--- Kết quả Test ---
MAE Test:   1266.6508
MSE Test:   7372024.5505
RMSE Test:  2715.1472
R2 Test:    0.5335

Thời gian Train: 75.9276 giây

========== FOLD 2 ==========
Số mẫu Train: 43020
Số mẫu Test: 10755

--- Kết quả Train ---
MAE Train:  1278.1612
MSE Train:  7432750.9342
RMSE Train: 2726.3072
R2 Train:   0.5345

--- Kết quả Test ---
MAE Test:   1252.4961
MSE Test:   7154040.6616
RMSE Test:  2674.7038
R2 Test:    0.5403

Thời gian Train: 100.5999 giây

========== FOLD 3 ==========
Số mẫu Train: 43020
Số mẫu Test: 10755

--- Kết quả Train ---
MAE Train:  1276.3910
MSE Train:  7386373.9653
RMSE Train: 2717.7884
R2 Train:   0.5353

--- Kết quả Test ---
MAE Test:   1259.0002
MSE Test:   7283287.6285
RMSE Test:  2698.7567
R2 Test:    0.5407

Thời gian Train: 105.2777 giây

========== FOLD 4 ==========
Số mẫ